# ML-04 — Search Intelligence Data Contract

This executed notebook uses the real March 2026 warehouse partition. March is a mid-panel development month; June is not used.


## 1. Unit of analysis + time window

1. **One row** is one pseudonymized client-content item on one reporting date.
2. **Table**: `fact_content_daily_performance`, March 2026 partition.
3. **Window**: 1–31 March 2026; features are at the end of day *t* and the proxy label is a click on *t+1*. The final date is not scored.
4. **Predict/rank**: the chance an item receives one or more GSC clicks tomorrow, as a near-term search-activity proxy.
5. **Exclude**: hashes (context only) and all tomorrow/outcome fields, which would identify an item or leak the result.


In [1]:
from pathlib import Path
import duckdb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# This cached path is created by a normal authenticated HF download; no secret is stored here.
paths = list((Path.home()/'.cache'/'huggingface'/'hub').rglob('fact_content_daily_performance/month=2026-03/data_0.parquet'))
safe_path = str(paths[0]).replace("'", "''")
if not paths: raise RuntimeError('Download the March partition with an HF READ token first; never paste a token into this notebook.')
con = duckdb.connect()
# con.execute('CREATE VIEW march AS SELECT * FROM read_parquet(?)', [str(paths[0])])
con.execute(f"CREATE VIEW march AS SELECT * FROM read_parquet('{safe_path}')")
print('March 2026 partition opened.')

March 2026 partition opened.


## 2. Fields: feature / label / context / excluded

- **Features (five):** today’s GSC impressions, clicks, average position, CTR calculated from those two quantities, and calendar day-of-week.
- **Label/proxy:** `next_day_has_click`, calculated from the next date’s `gsc_clicks > 0` for the same client-content item.
- **Context:** date, client hash, content hash, and `gsc_data_available`; keys define the panel but are never model inputs.
- **Excluded:** tomorrow’s clicks and every other tomorrow metric (label leakage); `ga4_*` (availability differs by client/date); hashes (not generalizable features).


In [2]:
# Exactly three verification queries.
q1='''SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_key FROM march GROUP BY 1,2,3 HAVING COUNT(*)>1 LIMIT 5'''
r1=con.sql(q1).df(); 
print('Query 1 — duplicate client-content-date keys:',len(r1)); 
display(r1)

q2='''SELECT COUNT(*) AS rows_in_slice, MIN(report_date) AS first_date, MAX(report_date) AS last_date FROM march'''
r2=con.sql(q2).df(); 
print('Query 2 — row count and date span:'); 
display(r2)

q3='''SELECT COUNT(*) AS all_march_rows, COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_gsc_is_true FROM march'''
r3=con.sql(q3).df(); 
print('Query 3 — availability, explicitly using IS TRUE:'); 
display(r3)


Query 1 — duplicate client-content-date keys: 0


,report_date,client_hash_id,content_hash_id,rows_at_key


Query 2 — row count and date span:


,rows_in_slice,first_date,last_date
0,9841378,2026-03-01,2026-03-31


Query 3 — availability, explicitly using IS TRUE:


,all_march_rows,rows_surviving_gsc_is_true
0,9841378,3611061


## 3. Five-feature frame and the trap

- `log_impressions_t` — knowable because today’s GSC impressions are reported before tomorrow is predicted.
- `log_clicks_t` — knowable because today’s clicks have closed before tomorrow.
- `gsc_avg_position_t` — knowable because it is today’s reported GSC ranking summary.
- `ctr_t` — knowable because it uses only today’s known clicks and impressions.
- `day_of_week` — knowable because the date is known in advance.

The code intentionally copies the label into a leaky column, shows its score, then deletes it and preserves the honest score.


In [3]:
sql='''WITH p AS (SELECT report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,NULLIF(gsc_avg_position,0) pos,LEAD(gsc_clicks) OVER(PARTITION BY client_hash_id,content_hash_id ORDER BY report_date) nxt FROM march WHERE gsc_data_available IS TRUE) SELECT log(1+gsc_impressions) log_impressions_t,log(1+gsc_clicks) log_clicks_t,COALESCE(pos,100.0) gsc_avg_position_t,COALESCE(gsc_clicks::DOUBLE/NULLIF(gsc_impressions,0),0.0) ctr_t,dayofweek(report_date) day_of_week,(nxt>0)::INTEGER next_day_has_click FROM p WHERE report_date<DATE '2026-03-31' AND nxt IS NOT NULL USING SAMPLE 250000 ROWS (reservoir,202603)'''
f=con.sql(sql).df(); 

names=['log_impressions_t','log_clicks_t','gsc_avg_position_t','ctr_t','day_of_week']; 
print('Feature frame:',f.shape); display(f.head())

cut=int(len(f)*.8); 
tr,te=f.iloc[:cut],f.iloc[cut:]; 

m=LogisticRegression(max_iter=200,random_state=202603).fit(tr[names],tr.next_day_has_click); 
honest=roc_auc_score(te.next_day_has_click,m.predict_proba(te[names])[:,1])
leak='leaked_next_day_has_click'; 

tr2,te2=tr.copy(),te.copy(); 
tr2[leak]=tr2.next_day_has_click; 
te2[leak]=te2.next_day_has_click; 

lm=LogisticRegression(max_iter=200,random_state=202603).fit(tr2[names+[leak]],tr2.next_day_has_click); 
leaky=roc_auc_score(te2.next_day_has_click,lm.predict_proba(te2[names+[leak]])[:,1])

print(f'AUC with deliberate label leak: {leaky:.4f}'); 
print(f'Honest AUC after deleting it: {honest:.4f}'); 
del tr2[leak],te2[leak]; 
print('Leak column deleted; allowed features remain:',names)


Feature frame: (237638, 6)


,log_impressions_t,log_clicks_t,gsc_avg_position_t,ctr_t,day_of_week,next_day_has_click
0,1.431364,0.0,18.884615,0.0,5,0
1,1.041393,0.0,0.600000,0.0,4,0
2,0.845098,0.0,42.833333,0.0,5,0
3,1.278754,0.0,12.055556,0.0,6,1
4,1.544068,0.0,22.323529,0.0,5,0


AUC with deliberate label leak: 1.0000
Honest AUC after deleting it: 0.8741
Leak column deleted; allowed features remain: ['log_impressions_t', 'log_clicks_t', 'gsc_avg_position_t', 'ctr_t', 'day_of_week']


## 4. Data limits

**Named limitation:** this is an unbalanced client-content panel. Missing earlier rows can mean the client’s history began later, not zero performance; filtering to GSC availability also changes the population. It supports observed next-day ranking, not causal claims.

## 5. Self-check

- Contract, three executed checks, five features, deliberate leak removed, and a limitation are included.
